# Music Generation II

In [152]:
import os
import pathlib
import statistics
from datetime import datetime
from statistics import mean
from typing import Protocol, Any, Tuple

import math
import musiclang
import pandas as pd
from musiclang_predict import MusicLangPredictor
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

In [153]:
current_dir = pathlib.Path.cwd()
data_path = current_dir.joinpath("data")
output_path = data_path.joinpath("output")

N_SCORES_TO_GENERATE = 10
ROOT_SEED = 0


def get_seeds(n_seeds: int, seed: int = ROOT_SEED) -> list[int]:
    return list(range(seed, n_seeds + seed))

## Exercise 1 [Generation with MusicLang, 4 points]

In [154]:
def get_score_name(seed: int) -> str:
    return f"score{seed}"


def generate_n_scores(n_scores: int = N_SCORES_TO_GENERATE, init_seed: int = ROOT_SEED, verbose: bool = False,
                      save_files: bool = True):
    scores = []

    ml = MusicLangPredictor('musiclang/musiclang-v2')  # Only available model for now

    # Folder to store pieces
    # OUT PATH
    if save_files:
        out_folder_name = f"{datetime.now().strftime('%y%m%d-%H%M%S')}"
        out_folder_path = output_path.joinpath(out_folder_name)
        if not os.path.exists(out_folder_path):
            os.mkdir(out_folder_path)
    else:
        out_folder_path = None

    # Generation params
    nb_tokens = 1300  # ~ 30s of music (depending of the number of instruments generated)
    temperature = 0.9  # Don't go over 1.0, at your own risks !
    top_p = 1.0  # <=1.0, Usually 1 best to get not too much repetitive music

    for seed in get_seeds(n_seeds=n_scores, seed=init_seed):
        if verbose:
            print(f"Generating score for seed {seed}", end="")
        score = ml.predict(
            nb_tokens=nb_tokens,
            temperature=temperature,
            topp=top_p,
            rng_seed=seed
        )
        if verbose:
            print("DONE!")

        # Store score
        if save_files:
            midi_path = out_folder_path.joinpath(f"{get_score_name(seed)}.mid")
            score.to_midi(str(midi_path.absolute()))

        scores.append(score)

    return out_folder_path, scores


In [155]:
#out_path, _ = generate_n_scores(n_scores=N_SCORES_TO_GENERATE, verbose=True)

#TODO: uncomment

## Exercise 2 [Evaluation, 4 points]

In [156]:
# Get generated MIDI files
generated_scores_path = output_path.joinpath("260131-122957")
generated_scores = {}
for midi_file in generated_scores_path.glob("*.mid"):
    generated_scores[midi_file.name] = musiclang.Score.from_midi(str(midi_file.absolute()))

print(f"{len(generated_scores)} generated midi scores processed")

10 generated midi scores processed


In [157]:
# Get real MIDI files
lakh_dataset_path = data_path.joinpath("LAKH-MIDI")

lakh_scores = {}
for folder in sorted(lakh_dataset_path.glob("*")):
    if folder.is_dir():
        midi_files = list(folder.glob("*.mid"))
        assert len(midi_files) == 1, "Only one file per track"
        lakh_scores[folder.name] = musiclang.Score.from_midi(str(midi_files[0].absolute()))

print(f"{len(lakh_scores)} LAKH midi scores processed")

10 LAKH midi scores processed


In [161]:
class EvaluationMethod(Protocol):

    @staticmethod
    def evaluate(score: musiclang.Score, normalize: bool) -> float:
        """Returns a scalar given a piece."""

    @staticmethod
    def feature_name() -> str:
        """Returns the name of the feature."""


"""
Events have this information...

{
'event_name': 'note_on',
'duration': 0.25,
'pitch': 21,
'offset': 0.0,
'velocity': 95,
'instrument': 'xylophone',
'silence': 0,
'pedal': None
}
"""


class DifferentPitches(EvaluationMethod):

    @staticmethod
    def evaluate(score: musiclang.Score, normalize: bool = True) -> float:
        """
        Count the overall number of different pitches used in the score.
        Returns the count of unique MIDI pitch values.
        """
        if normalize:
            return normalize_metric(DifferentPitches, score)

        unique_pitches = set()

        events = score.to_events()
        for event in events:
            unique_pitches.add(int(event["pitch"]))

        return len(unique_pitches)

    @staticmethod
    def feature_name() -> str:
        return "DP"


class PitchRange(EvaluationMethod):

    @staticmethod
    def evaluate(score: musiclang.Score, normalize: bool = True) -> float:
        """
        Calculate the pitch range as the distance between highest and lowest pitch.
        Returns the difference in semitones.
        """
        if normalize:
            return normalize_metric(PitchRange, score)

        events = score.to_events()

        min_pitch = events[0]["pitch"]
        max_pitch = events[0]["pitch"]

        for event in events:
            if event["pitch"] > max_pitch:
                max_pitch = event["pitch"]
            if event["pitch"] < min_pitch:
                min_pitch = event["pitch"]

        return max_pitch - min_pitch

    @staticmethod
    def feature_name() -> str:
        return "PR"


class NInstruments(EvaluationMethod):

    @staticmethod
    def evaluate(score: musiclang.Score, normalize: bool = False) -> float:
        """
        Count the number of different musical instruments used in the track.
        """
        if normalize:
            return normalize_metric(NInstruments, score)

        return float(len(score.instrument_names))

    @staticmethod
    def feature_name() -> str:
        return "NI"


class FourNoteSequences(EvaluationMethod):

    @staticmethod
    def evaluate(score: musiclang.Score, normalize: bool = True) -> float:
        if normalize:
            return normalize_metric(FourNoteSequences, score)

        instruments = score.instrument_names
        n_instruments = len(instruments)

        onsets = 0
        patterns = 0
        for instrument in instruments:
            part = score.get_instruments(instrument)
            events = part.to_events()  # Notes of a part/instrument
            onset_groups = {}
            for event in events:
                offset = event['offset']
                if offset not in onset_groups:
                    onset_groups[offset] = []
                onset_groups[offset].append(event['duration'])

            # Sort by onset time
            sorted_onsets = sorted(onset_groups.keys())

            # For each onset, take the average
            consecutive_onset_durations = [statistics.mean(onset_groups[onset]) for onset in sorted_onsets]

            # Create all possible 4-note rhythm patterns from consecutive onsets
            for i in range(len(consecutive_onset_durations) - 3):
                if consecutive_onset_durations[i] == consecutive_onset_durations[i + 1] == consecutive_onset_durations[
                    i + 2] == consecutive_onset_durations[i + 3]:
                    patterns += 1
            onsets += len(consecutive_onset_durations)

        possible_patterns = (onsets - 3 * n_instruments)  # In paper 5... but I don't understand the value
        if possible_patterns > 0:
            return possible_patterns / n_instruments
        else:
            return 0

    @staticmethod
    def feature_name() -> str:
        return "4NS"


class RhythmRange(EvaluationMethod):

    @staticmethod
    def evaluate(score: musiclang.Score, normalize: bool = True) -> float:
        if normalize:
            return normalize_metric(RhythmRange, score)
        events = score.to_events()

        min_duration = events[0]["duration"]
        max_duration = events[0]["duration"]

        for event in events:
            if event["duration"] > max_duration:
                max_duration = event["duration"]
            if event["duration"] < min_duration:
                min_duration = event["duration"]

        return max_duration - min_duration

    @staticmethod
    def feature_name() -> str:
        return "RR"


def normalize_metric(method: EvaluationMethod, score: musiclang.Score, n_seconds: int = 10) -> float:
    qn_duration = score.duration
    qn_section = n_seconds * 120 / 60  # Should be 20 for 10s

    sections = math.ceil(qn_duration / qn_section)

    measures = []
    for i in range(sections):
        start = i * qn_section
        end = start + qn_section
        if end > qn_duration:
            end = qn_duration
        section = score.get_score_between(start, end)
        measures.append(method.evaluate(section, normalize=False))

    return mean(measures)


EVAL_METHODS = [DifferentPitches, PitchRange, NInstruments, FourNoteSequences, RhythmRange]

In [162]:
# Get dataframe

# DF
TARGET_COLUMN = "label"
ID_COLUMN = "song_id"

GENERATED_LABEL = "generated"
REAL_LABEL = "real"


def get_features(score: musiclang.Score) -> dict[str, Any]:
    features = {}
    for method in EVAL_METHODS:
        features[method.feature_name()] = method.evaluate(score)
    return features


def get_feature_dataframe(scores: dict[str, musiclang.Score], label: str) -> pd.DataFrame:
    row_list = []

    for score_name, score in scores.items():
        print(f"Processing score: {score_name}")
        features = get_features(score)
        features[ID_COLUMN] = score_name
        features[TARGET_COLUMN] = label
        row_list.append(features)

    df = pd.DataFrame(row_list)
    df.set_index(ID_COLUMN, inplace=True)
    return df

In [163]:
generated_df = get_feature_dataframe(generated_scores, GENERATED_LABEL)
print(generated_df)

                   DP         PR   NI        4NS        RR      label
song_id                                                              
score0.mid  12.200000  21.700000  1.0  13.900000  0.787500  generated
score1.mid  23.666667  45.666667  5.0  18.133333  3.541667  generated
score3.mid  25.000000  42.000000  1.0  38.333333  1.361111  generated
score2.mid  15.571429  37.857143  2.0  20.500000  3.190476  generated
score6.mid  13.333333  33.166667  1.0  15.000000  2.291667  generated
score7.mid  25.000000  53.500000  7.0  13.107143  1.895833  generated
score5.mid  16.000000  32.000000  4.0  21.333333  2.694444  generated
score4.mid  11.500000  35.833333  4.0   9.833333  2.250000  generated
score9.mid  22.000000  47.333333  5.0  24.533333  1.458333  generated
score8.mid  15.333333  36.666667  5.0  16.600000  3.652778  generated


In [164]:
real_df = get_feature_dataframe(lakh_scores, REAL_LABEL)
print(real_df)

                   DP         PR    NI        4NS        RR label
song_id                                                          
Track00001  36.000000  56.562500   9.0  12.692535  2.039062  real
Track00002  23.315789  48.052632  13.0  11.413576  1.822368  real
Track00003  38.148148  55.666667   9.0  24.567637  2.449074  real
Track00004  24.181818  52.454545   9.0  15.208929  1.439394  real
Track00005  35.000000  72.227273  12.0  22.684432  2.482955  real
Track00006  30.538462  51.615385  15.0  11.895144  2.900641  real
Track00007  30.350000  54.800000  10.0  22.595833  2.241667  real
Track00008  31.500000  52.428571   7.0  27.112245  1.747024  real
Track00009  26.687500  56.750000  11.0  16.260938  2.348958  real
Track00010  30.600000  54.400000   5.0  19.062222  1.769444  real


## Exercise 3 [Detection of Generated Pieces, 2 points]

In [165]:
N_CLASSIFIERS = 10
CLASSIFIER_SEED = 42


def train_test_classifiers(generated_df: pd.DataFrame, real_df: pd.DataFrame, train_size: int = 5,
                           test_size: int = 5) -> Tuple[pd.DataFrame, float]:
    results_list = []
    avg_acc = 0.0

    for i, seed in enumerate(range(CLASSIFIER_SEED, CLASSIFIER_SEED + N_CLASSIFIERS)):
        #print(f"{i} / {N_CLASSIFIERS} - {100 * i / N_CLASSIFIERS:.2f}%")

        train_5g, test_5g = train_test_split(generated_df, train_size=train_size, test_size=test_size,
                                             random_state=seed)
        train_5r, test_5r = train_test_split(real_df, train_size=train_size, test_size=test_size, random_state=seed)
        train_df = pd.concat([train_5g, train_5r])
        test_df = pd.concat([test_5g, test_5r])

        X = train_df.drop(columns=[TARGET_COLUMN])
        Y = train_df[TARGET_COLUMN]

        clf = RandomForestClassifier(random_state=seed)
        clf = clf.fit(X, Y)

        x = test_df.drop(columns=[TARGET_COLUMN])
        y = test_df[TARGET_COLUMN]

        y_pred = clf.predict(x)

        acc = accuracy_score(y, y_pred)
        results_list.append({
            "model_name": seed,
            "accuracy": (acc),
            "f1_score": (f1_score(y, y_pred, average='weighted'))
        })
        avg_acc += acc

    avg_acc /= N_CLASSIFIERS

    return pd.DataFrame(results_list), avg_acc

In [166]:
results_img_df, avg_acc = train_test_classifiers(generated_df, real_df)

print(results_img_df, end="\n\n")

print(f"Average classifier accuracy: {avg_acc * 100:.2f}%")

   model_name  accuracy  f1_score
0          42       0.8  0.791667
1          43       1.0  1.000000
2          44       0.8  0.791667
3          45       1.0  1.000000
4          46       0.8  0.791667
5          47       0.9  0.898990
6          48       0.9  0.898990
7          49       0.7  0.670330
8          50       1.0  1.000000
9          51       0.9  0.898990

Average classifier accuracy: 88.00%
